# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begs44/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** My lane is a "which pages first?" ranking problem with an observed yes/no label (`trend_direction == "down"`, base rate 0.542). So I train classifiers and rank pages by their predicted probability, then score the top of the list with **precision@K**, the same metric as my Week-4 baseline.

I compare, from simple to strong:
- **Dummy**: the floor, which is the same as a random order.
- **Week-4 rule baseline**: CTR shortfall for position × log10(impressions), unchanged.
- **Logistic Regression**: readable; each feature gets one weight.
- **Decision Tree (depth 3)**: a tree small enough to print and read.
- **Gradient Boosting (HistGradientBoosting)**: stronger, handles missing values and non-linear patterns.

**Features:** 19 numeric and 3 categorical columns. They are 90-day totals (log-scaled), position, CTR, engagement, age, staleness, keyword data, and type/intent. `avg_position = 0` is turned into "missing", and missing word count and keyword data get their own has_ flags instead of 0. **Excluded:** `trend_direction`, `trend_pct` and all `*_last_30d` / `*_prev_30d` columns (the label is built from these), and the IDs. A guard in the code stops the run if any of them sneak in. The seed is fixed at 42.

In [1]:
import pandas as pd
import numpy as np
import sklearn

SEED = 42
url = "https://raw.githubusercontent.com/begs44/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["target_decay"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,} | clients: {df['client_id'].nunique()} | base rate: {df['target_decay'].mean():.3f} | sklearn {sklearn.__version__}\n")

# ---------- Features: only pre-decision, label-free columns ----------
df["avg_position"] = df["avg_position"].replace(0, np.nan)          # 0 = "no data", not rank zero
df["has_word_count"] = df["word_count"].notna().astype(int)         # missingness flags instead of fillna(0)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)

NUMERIC = ["search_volume", "competition", "cpc", "word_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
           "has_word_count", "has_keyword_data"]
CATEGORICAL = ["content_type", "main_intent", "competition_level"]
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])
df[CATEGORICAL] = df[CATEGORICAL].fillna("unknown")

FORBIDDEN = {"trend_direction", "trend_pct", "target_decay", "is_declining_label", "content_id", "client_id",
             "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
             "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"}
assert not FORBIDDEN & set(NUMERIC + CATEGORICAL), "LEAK in feature list"
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical features. Leakage guard passed.")
print("Models compared: Dummy (floor) | Week-4 rule baseline | Logistic Regression | Decision Tree (depth 3) | Gradient Boosting")

Rows: 30,000 | clients: 32 | base rate: 0.542 | sklearn 1.6.1

19 numeric + 3 categorical features. Leakage guard passed.
Models compared: Dummy (floor) | Week-4 rule baseline | Logistic Regression | Decision Tree (depth 3) | Gradient Boosting


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped 5-fold cross-validation by `client_id`.** Every client is either fully in train or fully in test. This is the honest question for FlyRank: *will the ranking work on a client the model has never seen?* A random split would let the model learn each client's own decline habits and then get tested on the same clients, which would inflate the score.

Why 5 folds instead of one split: there are only 32 clients and they are very unequal in size. The largest has 7,008 pages, and it fills fold 1 on its own. One single split could be lucky or unlucky, so I report the **mean and spread over 5 folds**.

The **baseline rule is scored on exactly the same test folds.** Its position-band CTR medians are computed from the training folds only.

Note: the Week-4 figure (P@50 = 0.64) ranked all 30,000 pages together. Here each fold ranks about 6,000 pages, so the numbers are not directly comparable. The fair comparison is the table in section 3.

In [2]:
from sklearn.model_selection import GroupKFold

# ---------- Grouped 5-fold split: every client is entirely in train OR in test ----------
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(df, df["target_decay"], groups=df["client_id"]))

rows = []
for i, (tr, te) in enumerate(folds, 1):
    tr_clients, te_clients = set(df.iloc[tr]["client_id"]), set(df.iloc[te]["client_id"])
    assert not tr_clients & te_clients, "client leaked across the split"
    rows.append({"fold": i, "train_rows": len(tr), "test_rows": len(te), "test_clients": len(te_clients),
                 "largest_test_client_share": round(df.iloc[te]["client_id"].value_counts(normalize=True).iloc[0], 2),
                 "test_base_rate": round(df.iloc[te]["target_decay"].mean(), 3)})
split_table = pd.DataFrame(rows).set_index("fold")
print(split_table.to_string())
print("\nNo client appears in both train and test in any fold.")

      train_rows  test_rows  test_clients  largest_test_client_share  test_base_rate
fold                                                                                
1          22992       7008             1                       1.00           0.490
2          24269       5731             7                       0.64           0.645
3          24247       5753             8                       0.40           0.379
4          24245       5755             8                       0.39           0.622
5          24247       5753             8                       0.31           0.585

No client appears in both train and test in any fold.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Result (mean over 5 client-grouped folds, test base rate 0.544):**

| Model | P@20 | P@50 | P@100 | ROC-AUC |
|---|---|---|---|---|
| Dummy (floor) | 0.61 | 0.568 | 0.558 | 0.500 |
| **Week-4 rule baseline** | 0.76 | **0.752** | 0.768 | 0.632 |
| Logistic Regression | 0.78 | 0.800 | 0.792 | 0.661 |
| Decision Tree (d=3) | 0.61 | 0.620 | 0.602 | 0.610 |
| Gradient Boosting | 0.86 | **0.808** | 0.782 | 0.678 |

- **Both Logistic Regression and Gradient Boosting beat the baseline,** by about 5 points at P@50 and 3–4 points at ROC-AUC.
- **Gradient Boosting vs Logistic Regression is a tie.** 0.808 vs 0.800 at P@50 is far smaller than the fold-to-fold spread (sd ≈ 0.07–0.10), and LR even wins at P@100. Complexity alone is not rewarded here. LR is the simpler model and nearly as good.
- **The depth-3 tree loses to the rule baseline.** Eight leaves are too coarse for ranking, because many pages share the same score.
- **The gain is not the same everywhere.** Gradient Boosting wins clearly on the biggest client (fold 1: 0.92 vs 0.66), but loses to the baseline on fold 2 (0.76 vs 0.84).

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def prep(scale):
    num = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()) if scale else SimpleImputer(strategy="median")
    return make_column_transformer((num, NUMERIC),
                                   (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))

models = {
    "Dummy (floor)":         lambda: DummyClassifier(strategy="prior"),
    "Logistic Regression":   lambda: make_pipeline(prep(True), LogisticRegression(max_iter=2000, C=0.5)),
    "Decision Tree (d=3)":   lambda: make_pipeline(prep(False), DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEED)),
    "Gradient Boosting":     lambda: make_pipeline(prep(False), HistGradientBoostingClassifier(
                                         max_depth=4, learning_rate=0.05, max_iter=300, min_samples_leaf=100, random_state=SEED)),
}

def baseline_rule_score(train, test):
    """Week-4 rule, unchanged: CTR shortfall vs position-band median x log10(impressions). Medians from TRAIN only."""
    bands = [0, 3, 5, 10, 15, 20]
    vis = lambda d: (d["impressions_90d"] >= 500) & (d["avg_position"] > 0) & (d["avg_position"] <= 20)
    exp = train[vis(train)].groupby(pd.cut(train.loc[vis(train), "avg_position"], bands), observed=True)["ctr"].median()
    e = pd.cut(test["avg_position"], bands).map(exp).astype(float)
    shortfall = (1 - test["ctr"] / e).clip(0, 1)
    return np.where(vis(test), shortfall * np.log10(test["impressions_90d"]), 0.0) + 1e-9 * test["impressions_90d"].values

def p_at_k(y, s, k):
    return float(np.asarray(y)[np.argsort(-np.asarray(s), kind="stable")[:k]].mean())

X, y = df[NUMERIC + CATEGORICAL], df["target_decay"]
df["oof_gb"] = np.nan
df["fold"] = 0
gb_models = {}
results = []
for i, (tr, te) in enumerate(folds, 1):
    y_te = y.iloc[te]
    scores = {"Week-4 rule baseline": baseline_rule_score(df.iloc[tr], df.iloc[te])}
    for name, make in models.items():
        m = make().fit(X.iloc[tr], y.iloc[tr])
        scores[name] = m.predict_proba(X.iloc[te])[:, 1]
        if name == "Gradient Boosting":
            gb_models[i] = m
    df.loc[df.index[te], "oof_gb"] = scores["Gradient Boosting"]
    df.loc[df.index[te], "fold"] = i
    for name, s in scores.items():
        results.append({"fold": i, "model": name, "P@20": p_at_k(y_te, s, 20), "P@50": p_at_k(y_te, s, 50),
                        "P@100": p_at_k(y_te, s, 100),
                        "ROC-AUC": roc_auc_score(y_te, s), "PR-AUC": average_precision_score(y_te, s)})

res = pd.DataFrame(results)
order = ["Dummy (floor)", "Week-4 rule baseline", "Logistic Regression", "Decision Tree (d=3)", "Gradient Boosting"]
mean = res.groupby("model")[["P@20", "P@50", "P@100", "ROC-AUC", "PR-AUC"]].mean().loc[order]
std = res.groupby("model")[["P@50", "ROC-AUC"]].std().loc[order].add_suffix(" sd")
table = mean.join(std).round(3)
print(f"Grouped 5-fold CV, mean over folds. Test base rate (mean): {split_table['test_base_rate'].mean():.3f}\n")
print(table.to_string(), "\n")
print("P@50 per fold:")
print(res.pivot(index="model", columns="fold", values="P@50").loc[order].round(2).to_string())

Grouped 5-fold CV, mean over folds. Test base rate (mean): 0.544

                      P@20   P@50  P@100  ROC-AUC  PR-AUC  P@50 sd  ROC-AUC sd
model                                                                         
Dummy (floor)         0.61  0.568  0.558    0.500   0.544    0.129       0.000
Week-4 rule baseline  0.76  0.752  0.768    0.632   0.652    0.087       0.095
Logistic Regression   0.78  0.800  0.792    0.661   0.674    0.095       0.033
Decision Tree (d=3)   0.61  0.620  0.602    0.610   0.610    0.091       0.076
Gradient Boosting     0.86  0.808  0.782    0.678   0.683    0.070       0.041 

P@50 per fold:
fold                     1     2     3     4     5
model                                             
Dummy (floor)         0.46  0.66  0.40  0.64  0.68
Week-4 rule baseline  0.66  0.84  0.66  0.78  0.82
Logistic Regression   0.80  0.86  0.64  0.82  0.88
Decision Tree (d=3)   0.50  0.60  0.58  0.72  0.70
Gradient Boosting     0.92  0.76  0.74  0.80  0.82


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on (permutation importance on held-out folds):**
1. **`days_with_impressions`** is by far the top feature. It **makes sense, but it also needs a warning.** A page with impressions on fewer than 30 days is often "new" or "flat" (it had nothing to lose last month), so it can hardly be labelled "down". The feature partly encodes *whether the page can decline at all*. To check this, I retrained Gradient Boosting **without** the day-count columns: P@50 drops only from 0.808 to 0.796, still above the baseline's 0.752. So the win does not depend on this feature.
2. **`avg_position`** comes second. Pages near the top have the most to lose.
3. **`content_age_days`** comes third. Young pages decline more, which matches the OPPOSITE verdict in my signal audit.

`ctr` and `log_impressions_90d` come right after. These are the same signals my rule uses, so the model keeps my baseline's idea and adds to it.

**Where it is wrong:**
- In the top-50 of each fold, 202 of 250 pages were hits (0.81).
- The 48 misses look like the hits but sit slightly **lower** on page 1 (median position 7.4 vs 3.6), with a bit more CTR (0.05% vs 0.00%).
- Precision of the top-20% flags is fairly even across visibility levels (0.67–0.74). It is weakest on tiny pages (< 100 impressions), where one month of noise can flip the label.

**Three confident misses (scored 0.95–0.96, not declining):** all three are pages at position ~2 with 2,000–4,800 impressions, CTR 0.06–0.15%, about 140 days old, updated 104 days ago, and with long articles (5,600–7,000 words). On paper they look exactly like decaying pages. They are hard because the label only compares two 30-day windows: a strong page that simply held steady this month looks identical to one that is about to slip.

**Caveats:** the 90-day totals overlap the label's 30-day windows (the same caveat as my baseline). The results come from one 30k-row snapshot with 32 clients. These are observed, directional results for decision support, not causal claims.

In [4]:
from sklearn.inspection import permutation_importance

# ---------- What does Gradient Boosting lean on? Permutation importance on each held-out fold ----------
imps = []
for i, (tr, te) in enumerate(folds, 1):
    r = permutation_importance(gb_models[i], X.iloc[te], y.iloc[te], scoring="average_precision",
                               n_repeats=3, random_state=SEED)
    imps.append(pd.Series(r.importances_mean, index=X.columns))
imp = pd.concat(imps, axis=1)
imp_table = pd.DataFrame({"mean_drop_PR_AUC": imp.mean(axis=1), "folds_positive": (imp > 0).sum(axis=1)})
print("Permutation importance (drop in PR-AUC when the column is shuffled), top 8:")
print(imp_table.sort_values("mean_drop_PR_AUC", ascending=False).head(8).round(4).to_string(), "\n")

# ---------- Where is it wrong? Top-50 of each test fold ----------
top = df.sort_values("oof_gb", ascending=False).groupby("fold").head(50).copy()
top["outcome"] = np.where(top["target_decay"] == 1, "hit (declining)", "miss (not declining)")
cols = ["impressions_90d", "ctr", "avg_position", "content_age_days", "days_with_impressions", "word_count"]
print("Top-50 per fold (250 pages) — medians, hits vs misses:")
print(top.groupby("outcome")[cols].median().round(2).to_string())
print(top["outcome"].value_counts().to_string(), "\n")

# Error rate across all held-out pages by visibility (model flags = top 20% score within its fold)
df["flag"] = df.groupby("fold")["oof_gb"].rank(pct=True) >= 0.8
vis_b = pd.cut(df["impressions_90d"], [0, 100, 500, 3000, 1e12], labels=["<100", "100-499", "500-2,999", "3,000+"], right=False)
err = df[df["flag"]].groupby(vis_b[df["flag"]], observed=True)["target_decay"].agg(n="size", precision="mean").round(3)
print("Precision of model's top-20% flags, by visibility bucket:")
print(err.to_string(), "\n")

# Three concrete wrong cases: the highest-scored pages that were NOT declining
wrong = df[df["target_decay"] == 0].sort_values("oof_gb", ascending=False).head(3)
print("3 most confident misses (no IDs shown):")
print(wrong[["oof_gb"] + cols + ["days_since_last_update", "main_intent"]].round(3).to_string(index=False))

# ---------- Sanity check on the top feature: does the win survive without the day-count columns? ----------
DROP = ["days_with_impressions", "days_with_sessions"]
num_red = [c for c in NUMERIC if c not in DROP]
abl = []
for i, (tr, te) in enumerate(folds, 1):
    pre = make_column_transformer((SimpleImputer(strategy="median"), num_red),
                                  (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
    m = make_pipeline(pre, HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
                                                          min_samples_leaf=100, random_state=SEED))
    m.fit(df.iloc[tr][num_red + CATEGORICAL], y.iloc[tr])
    s = m.predict_proba(df.iloc[te][num_red + CATEGORICAL])[:, 1]
    abl.append({"P@50": p_at_k(y.iloc[te], s, 50), "ROC-AUC": roc_auc_score(y.iloc[te], s)})
abl = pd.DataFrame(abl).mean().round(3)
print(f"\nGradient Boosting WITHOUT day-count features: P@50 = {abl['P@50']}, ROC-AUC = {abl['ROC-AUC']}")
print(f"(with them: P@50 = {table.loc['Gradient Boosting', 'P@50']}, ROC-AUC = {table.loc['Gradient Boosting', 'ROC-AUC']}; "
      f"rule baseline P@50 = {table.loc['Week-4 rule baseline', 'P@50']})")

Permutation importance (drop in PR-AUC when the column is shuffled), top 8:
                       mean_drop_PR_AUC  folds_positive
days_with_impressions            0.0903               5
avg_position                     0.0295               4
content_age_days                 0.0164               5
ctr                              0.0162               5
log_impressions_90d              0.0147               5
scroll_rate                      0.0110               5
days_with_sessions               0.0037               4
log_clicks_90d                   0.0036               4 

Top-50 per fold (250 pages) — medians, hits vs misses:
                      impressions_90d   ctr  avg_position  content_age_days  days_with_impressions  word_count
outcome                                                                                                       
hit (declining)                1351.0  0.00          3.55             147.0                   80.0      2876.0
miss (not declining)          

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.